# Lasmoid: 10M Parameter Model Training Pipeline on Kaggle

This notebook demonstrates how to clone, configure, and train a **10M parameter version** of the **Lasmoid Hybrid Concept Transformer** model on a Kaggle GPU instance (e.g. T4 GPU or P100).

### Core Architectural Features in this 10M Config:
- **Shared Embedding/Output Projection Weight Tying**: Cuts parameter footprint by sharing vocab matrices.
- **Interleaved Attention/SSM Layers**: Interleaves CSA (Compressed Sparse Attention) with parallel State Space Recurrence (Mamba-2/SSD).
- **Manifold-Constrained Hyper-Connections (mHC)**: Routes features over residual streams using projected doubly stochastic weight matrices.
- **Muon Optimizer**: Newton-Schulz 5th-order orthogonalization for 2D weights, combined with `AdamW` for 1D scaling, biases, and embedding tables.

## Step 1: Environment Setup & Codebase Retrieval

In [1]:
# Install dependencies (Kaggle environments already have PyTorch installed)
!pip install -q transformers datasets tqdm

import os, shutil, sys

import torch, warnings
if not hasattr(torch, 'bf16'):
    torch.bf16 = torch.bfloat16
warnings.filterwarnings('ignore', message='.*incorrect regex pattern.*')
from pathlib import Path

# Start absolutely in /kaggle/working
os.chdir("/kaggle/working")

REPO_URL = "https://github.com/Theory903/Lasmoid.git"
REPO_DIR = Path("/kaggle/working/Lasmoid")

# Clean up nested Lasmoid folders if they exist
nested_dir = REPO_DIR / "Lasmoid"
while nested_dir.exists():
    print("🧹 Found nested repository copy. Cleaning up...")
    shutil.rmtree(nested_dir, ignore_errors=True)

# Clone or pull cleanly
if not REPO_DIR.exists():
    print("Cloning Lasmoid repository...")
    os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
else:
    if not (REPO_DIR / ".git").exists():
        print(f"Directory {REPO_DIR} exists but is not a valid git repository. Re-cloning...")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    else:
        print("Updating Lasmoid repository...")
        os.system(f"git -C {REPO_DIR} fetch --all")
        os.system(f"git -C {REPO_DIR} reset --hard origin/main")

# Change directory absolutely
os.chdir(str(REPO_DIR))

# Ensure correct sys.path ordering
repo_paths = [str(REPO_DIR), str(REPO_DIR / "inference"), str(REPO_DIR / "train")]
for p in reversed(repo_paths):
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

print(f"Current Directory: {os.getcwd()}")
print("✅ Codebase retrieved and prepared.")


Cloning into 'Lasmoid'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 36 (delta 6), reused 34 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 1.90 MiB | 6.15 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/Lasmoid


## Step 2: Define the 10M Parameter Configuration

By setting `dim = 64` and `n_layers = 2` with weight tying enabled for the `129,286` size vocabulary, the embedding/head weights consume `8.27M` parameters, leaving roughly `2M` parameters for the hybrid attention, Mamba recurrence, and MoE routing layers, summing to a **10.3M parameter model**.

In [2]:
import json

config_10m = {
  "vocab_size": 129280,
  "dim": 64,
  "n_layers": 2,
  "max_seq_len": 128,
  "max_batch_size": 8,
  "dtype": "bf16",
  "scale_fmt": None,
  "scale_dtype": "fp32",
  "expert_dtype": None,
  "norm_eps": 1e-06,
  "n_heads": 4,
  "q_lora_rank": 16,
  "head_dim": 16,
  "rope_head_dim": 8,
  "o_groups": 2,
  "o_lora_rank": 16,
  "rope_theta": 10000.0,
  "rope_factor": 1.0,
  "beta_fast": 32,
  "beta_slow": 1,
  "original_seq_len": 0,
  "n_routed_experts": 4,
  "n_shared_experts": 1,
  "n_activated_experts": 2,
  "n_group": 1,
  "topk_group": 1,
  "moe_inter_dim": 0,
  "moe_latent_dim": 32,
  "score_func": "sqrtsoftplus",
  "route_scale": 1.0,
  "swiglu_limit": 10.0,
  "n_hash_layers": 0,
  "num_residual_streams": 4,
  "hc_sinkhorn_iters": 8,
  "hc_eps": 1e-06,
  "num_concepts": 16,
  "num_abstract_concepts": 4,
  "num_global_concepts": 2,
  "n_mtp_layers": 1,
  "window_size": 64,
  "codebook_size": 128,
  "hcm_ema_alpha": 0.99,
  "hcm_commit_loss_coeff": 0.25,
  "entropy_threshold": 0.5,
  "lightning_topk_blocks": 1,
  "router_z_loss_coeff": 0.001,
  "ema_bias_lr": 0.01,
  "expert_capacity_factor": 1.25,
  "moe_load_balance_coeff": 0.01,
  "predictive_coding_coeff": 0.01,
  "ssm_heads": 2,
  "ssm_state_dim": 8,
  "ssm_kernel_size": 4,
  "ssm_chunk_size": 16,
  "ssm_dt_min": 0.001,
  "ssm_dt_max": 0.1,
  "ssm_dt_init_floor": 0.0001,
  "ssm_n_groups": 1,
  "reasoning_steps": 1,
  "think_token_id": 128821,
  "answer_token_id": 129285,
  "cot_exit_confidence": 0.9,
  "external_embedding_dim": 0,
  "external_embedding_scale": 0.25,
  "external_embedding_norm": True,
  "moe_router_entropy_coeff": 0.001,
  "moe_capacity_loss_coeff": 0.01,
  "token_concept_loss_coeff": 0.05,
  "steering_attributes": [
    "creativity",
    "helpfulness",
    "complexity",
    "scientific_rigor"
  ]
}

# Overwrite config.json with the 10M parameter version
with open("config.json", "w") as f:
    json.dump(config_10m, f, indent=2)

print("config.json updated successfully for 10M parameter version.")

config.json updated successfully for 10M parameter version.


## Step 3: Verify the Parameter Count

Let's run the model initialization script to verify that the parameter count matches our ~10M expectation and that all shapes align.

In [3]:
import torch
from inference.model import Lasmoid, ModelArgs

with open("config.json") as f:
    config_dict = json.load(f)

args = ModelArgs(**config_dict)
model = Lasmoid(args)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model Dimension: {args.dim}")
print(f"Number of Layers: {args.n_layers}")
print(f"Model Parameters: {total_params:,}")

Model Dimension: 64
Number of Layers: 2
Model Parameters: 8,642,835


## Step 4: Run Training

We will run the training loop for 500 iterations on the GPU (`cuda`). The dataset (`input.txt`) will be downloaded automatically by the script.

In [4]:
# Run training loop on Kaggle GPU
!python train/train.py --max_iters 500 --batch_size 8 --learning_rate 6e-4 --device cuda

Using device: CUDA (DDP: False)
Loading text dataset from: input.txt
Model initialized with dim=64, layers=2, heads=4, vocab_size=129280
Training initiated...
Step    0 | Total Loss: 15.3230 | CE (t+1): 11.7670 | MTP (t+2): 11.7780 | VQ Loss: 0.0000 | Pred Loss: 0.0000 | LR Scale: 0.0000
Step   20 | Total Loss: 14.7224 | CE (t+1): 11.3029 | MTP (t+2): 11.3046 | VQ Loss: 0.0000 | Pred Loss: 0.0000 | LR Scale: 1.0000
Step   40 | Total Loss: 13.2109 | CE (t+1): 10.1395 | MTP (t+2): 10.1443 | VQ Loss: 0.0000 | Pred Loss: 0.0000 | LR Scale: 0.9957
Step   60 | Total Loss: 11.6655 | CE (t+1): 8.9599 | MTP (t+2): 8.9609 | VQ Loss: 0.0000 | Pred Loss: 0.0000 | LR Scale: 0.9830
Step   80 | Total Loss: 10.5725 | CE (t+1): 8.1107 | MTP (t+2): 8.1090 | VQ Loss: 0.0000 | Pred Loss: 0.0000 | LR Scale: 0.9619
Step  100 | Total Loss: 9.7931 | CE (t+1): 7.5231 | MTP (t+2): 7.5310 | VQ Loss: 0.0000 | Pred Loss: 0.0000 | LR Scale: 0.9330
Checkpoint saved to checkpoints/current/lasmoid_step_100.pt
Step  12

## Step 5: Test Text Generation

Once training completes, we will test the text generator using our trained checkpoint.

In [5]:
with open("test_prompts.txt", "w") as f:
    f.write("The cat sat on the mat\n")

# Run generation in batch mode
!python inference/generate.py --ckpt-path checkpoints/current --config config.json --input-file test_prompts.txt --max-new-tokens 30 --device cuda

[generate] Device: CUDA
[generate] Found checkpoint: checkpoints/current/lasmoid_step_100.pt
[generate] Loaded config from config.json
[generate] Successfully loaded weights from checkpoint.
[generate] Batch mode: 1 prompts
Prompt:     The cat sat on the mat
Completion:  hath a their this on say that.
IO usOL forTo. to,,KEAnd beWith,
 myTo are in of you give,
──────────────────────────────────────────────────
